PAC Report — Analytical Process & Learning Summary

1. Introduction

This project focused on predicting monthly credit card spending using demographic, financial, and behavioral attributes. My modeling process evolved across multiple submissions (pac1–pac11), gradually improving through trial, error, and iterative refinement. This report summarizes what I learned from exploring the data, preparing features, and experimenting with different modeling techniques before arriving at the final model.

2. Data Exploration

Initial exploration revealed that transaction-related variables—such as the number of transactions, average transaction value, and online shopping frequency—were the strongest indicators of monthly spend. Income and credit limit also carried signal but showed wide variation. In contrast, several demographic variables contributed little predictive value. I also discovered that monthly_spend was heavily right-skewed, suggesting that later models would benefit from a log transformation.

Key takeaway: strong numeric predictors matter more than broad sets of weak categorical variables.

3. Early Modeling Attempts

My early models (pac1–pac3) used simple linear regression with only a few features. These models underfit badly and made it clear that spending behavior involves nonlinear relationships. Although performance was poor, the early attempts provided a useful baseline.

What went wrong: too few features, overly simple models, and no structure around validation.

4. Feature Preparation & Evolving Techniques

As I added more predictors and introduced categorical encoding (pac4–pac6), performance improved somewhat, but I also began to experience issues with noise and overly large feature spaces. Later, I applied a log transformation to the target and reduced unnecessary categorical encoding. These changes produced more stable models.

What I learned: feature engineering should be intentional, not exhaustive, and transformations can significantly improve model behavior.

5. Advancing Models Through Iteration

In pac7–pac10, I experimented with Gradient Boosting, log-transformed targets, and early hyperparameter tuning. These models captured complexity better, though pipeline inconsistency and occasional incomplete cells made the workflow fragile. By pac11, I consolidated the best ideas—selective features, consistent preprocessing, and tuned models—into a more stable approach.

What improved: validation, feature selection, and model sophistication.
What went wrong: overcomplicating pipelines before establishing a clean foundation.

6. Overall Lessons Learned

Throughout the iterative process:

Simpler, high-signal features consistently outperformed large sets of weak predictors.

Clean preprocessing pipelines are essential for reproducibility.

Validation must be systematic to compare models fairly.

Most improvements came from correcting mistakes—such as overusing categorical encoding or building overly complex models too early.

7. If I Had to Do It Again

I would begin with more thorough exploratory analysis, introduce log-transformed targets sooner, and build a simple but clean baseline model before testing more complex methods. I would also focus earlier on reducing noise and ensuring consistency between training and scoring pipelines.

8. Transition to Final Code

The final section of this notebook presents the well-documented code used for my best submission, reflecting the cumulative learning from all prior iterations.

In [1]:
# 0. Imports & Global Settings
import numpy as np
import pandas as pd
import warnings

from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

from sklearn.linear_model import Ridge, LassoCV, LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
N_SPLITS = 5  # K-fold stacking


In [2]:
# 1. Load Data
data = pd.read_csv("analysis_data.csv")
print("Training data shape:", data.shape)
display(data.head())


Training data shape: (40000, 23)


,customer_id,age,gender,marital_status,education_level,region,employment_status,owns_home,has_auto_loan,annual_income,...,card_type,num_transactions,avg_transaction_value,online_shopping_freq,reward_points_balance,travel_frequency,utility_payment_count,num_children,num_credit_cards,monthly_spend
0,75570383,77,female,married,high school,northeast,unemployed,1,0,56000.116893,...,standard,14,46.144967,8.0,6325.578928,3,3.0,1,8,1682.86
1,77915507,52,male,married,high school,south,unemployed,1,1,39432.589311,...,gold,13,88.532219,7.0,5586.814175,1,1.0,2,8,2135.57
2,31958910,48,female,married,bachelors,west,unemployed,1,1,51020.949673,...,standard,9,78.304445,4.0,6250.225115,3,2.0,0,8,1540.61
3,76868487,67,female,married,high school,northeast,unemployed,1,0,76794.566399,...,standard,10,54.932667,7.0,8875.409103,3,3.0,2,8,1250.86
4,69435006,21,female,single,bachelors,west,self-employed,1,1,44386.444631,...,standard,8,18.866712,6.0,5318.021523,4,3.0,0,8,915.36


In [3]:
# 2. Feature Engineering Function (Apply ONLY to predictors, no target leakage)
def add_features(df):
    df = df.copy()
    
    # Basic behavior-based features
    df["total_trans_value"] = df["num_transactions"] * df["avg_transaction_value"]
    df["income_credit_ratio"] = df["annual_income"] / (df["credit_limit"] + 1)
    df["online_trans_share"] = df["online_shopping_freq"] / (df["num_transactions"] + 1)
    df["income_per_person"] = df["annual_income"] / (df["num_children"] + 1)
    df["trips_per_transaction"] = df["travel_frequency"] / (df["num_transactions"] + 1)
    
    # Some interaction features
    df["credit_x_transactions"] = df["credit_limit"] * df["num_transactions"]
    df["income_x_avg_val"] = df["annual_income"] * df["avg_transaction_value"]
    df["transactions_sq"] = df["num_transactions"] ** 2
    
    return df

# Separate target and base features
y_raw = data["monthly_spend"]
X_base = data.drop(columns=["monthly_spend", "customer_id"])

# Apply feature engineering
X = add_features(X_base)

print("Feature-engineered X shape:", X.shape)
display(X.head())


Feature-engineered X shape: (40000, 29)


,age,gender,marital_status,education_level,region,employment_status,owns_home,has_auto_loan,annual_income,credit_score,...,num_children,num_credit_cards,total_trans_value,income_credit_ratio,online_trans_share,income_per_person,trips_per_transaction,credit_x_transactions,income_x_avg_val,transactions_sq
0,77,female,married,high school,northeast,unemployed,1,0,56000.116893,636.767610,...,1,8,646.029535,5.317448,0.533333,28000.058447,0.200000,147425.454638,2.584124e+06,196
1,52,male,married,high school,south,unemployed,1,1,39432.589311,739.726869,...,2,8,1150.918847,4.633643,0.500000,13144.196437,0.071429,110617.797674,3.491055e+06,169
2,48,female,married,bachelors,west,unemployed,1,1,51020.949673,650.995903,...,0,8,704.740006,4.093549,0.400000,51020.949673,0.300000,112164.690002,3.995167e+06,81
3,67,female,married,high school,northeast,unemployed,1,0,76794.566399,612.779395,...,2,8,549.326670,4.325978,0.636364,25598.188800,0.272727,177509.535766,4.218530e+06,100
4,21,female,single,bachelors,west,self-employed,1,1,44386.444631,714.429606,...,0,8,150.933696,4.500458,0.666667,44386.444631,0.444444,78893.212490,8.374263e+05,64


In [4]:
# Log-transformed target
y_log = np.log1p(y_raw)


In [5]:
# 3. Preprocessing: Impute + Encode + Scale

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


Numeric features: ['age', 'owns_home', 'has_auto_loan', 'annual_income', 'credit_score', 'credit_limit', 'tenure', 'num_transactions', 'avg_transaction_value', 'online_shopping_freq', 'reward_points_balance', 'travel_frequency', 'utility_payment_count', 'num_children', 'num_credit_cards', 'total_trans_value', 'income_credit_ratio', 'online_trans_share', 'income_per_person', 'trips_per_transaction', 'credit_x_transactions', 'income_x_avg_val', 'transactions_sq']
Categorical features: ['gender', 'marital_status', 'education_level', 'region', 'employment_status', 'card_type']


In [6]:
# 4. Base Models (trained on log(y))

base_models = {
    "rf": RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "gbrt": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=RANDOM_STATE,
    ),
    "hist_gbrt": HistGradientBoostingRegressor(
        max_depth=8,
        learning_rate=0.05,
        max_iter=400,
        random_state=RANDOM_STATE,
    ),
    "xgb": XGBRegressor(
        n_estimators=600,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",  # use "gpu_hist" if GPU available
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "lgbm": LGBMRegressor(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "cat": CatBoostRegressor(
        iterations=600,
        depth=6,
        learning_rate=0.05,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False,
    ),
}


In [7]:
# 5. K-Fold Out-of-Fold Stacking

kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

n_samples = X.shape[0]
base_oof_preds = {name: np.zeros(n_samples) for name in base_models.keys()}
base_cv_rmse = {}

# We'll also keep one fitted pipeline per model for later refitting
print("Starting K-fold OOF stacking...")

for name, model in base_models.items():
    print(f"\nModel: {name}")
    oof_pred = np.zeros(n_samples)
    fold_rmses = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_log_tr, y_log_val = y_log.iloc[train_idx], y_log.iloc[val_idx]
        y_raw_val = y_raw.iloc[val_idx]
        
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", clone(model)),
        ])
        
        pipe.fit(X_tr, y_log_tr)
        y_log_val_pred = pipe.predict(X_val)
        y_val_pred = np.expm1(y_log_val_pred)
        
        oof_pred[val_idx] = y_val_pred
        rmse = np.sqrt(mean_squared_error(y_raw_val, y_val_pred))
        fold_rmses.append(rmse)
        print(f"  Fold {fold} RMSE: {rmse:.4f}")
    
    base_oof_preds[name] = oof_pred
    mean_rmse = np.mean(fold_rmses)
    base_cv_rmse[name] = mean_rmse
    print(f"Mean CV RMSE for {name}: {mean_rmse:.4f}")


Starting K-fold OOF stacking...

Model: rf
  Fold 1 RMSE: 276.1461
  Fold 2 RMSE: 277.5001
  Fold 3 RMSE: 275.2208
  Fold 4 RMSE: 277.5226
  Fold 5 RMSE: 272.4807
Mean CV RMSE for rf: 275.7741

Model: gbrt
  Fold 1 RMSE: 258.3002
  Fold 2 RMSE: 260.5398
  Fold 3 RMSE: 257.8845
  Fold 4 RMSE: 260.9686
  Fold 5 RMSE: 255.3412
Mean CV RMSE for gbrt: 258.6069

Model: hist_gbrt
  Fold 1 RMSE: 260.4381
  Fold 2 RMSE: 261.3412
  Fold 3 RMSE: 258.0105
  Fold 4 RMSE: 261.4829
  Fold 5 RMSE: 255.6030
Mean CV RMSE for hist_gbrt: 259.3752

Model: xgb
  Fold 1 RMSE: 260.6597
  Fold 2 RMSE: 263.1739
  Fold 3 RMSE: 261.7246
  Fold 4 RMSE: 263.1593
  Fold 5 RMSE: 257.9519
Mean CV RMSE for xgb: 261.3339

Model: lgbm
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004009 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3322
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 41
[L

In [8]:
# 6. Train Stacking Meta-Learner (Ridge)

stack_model_names = list(base_models.keys())  # ['rf', 'gbrt', 'hist_gbrt', 'xgb', 'lgbm', 'cat']

# Meta features: OOF predictions from each base model
meta_X = np.column_stack([base_oof_preds[name] for name in stack_model_names])

stacker = Ridge(alpha=1.0)
stacker.fit(meta_X, y_raw)

stack_oof_pred = stacker.predict(meta_X)
stack_rmse = np.sqrt(mean_squared_error(y_raw, stack_oof_pred))

print("\n=== Stacking Performance ===")
print("Base models (OOF) RMSEs:")
for name, rmse in base_cv_rmse.items():
    print(f"  {name}: {rmse:.4f}")
print(f"Stacked model OOF RMSE: {stack_rmse:.4f}")



=== Stacking Performance ===
Base models (OOF) RMSEs:
  rf: 275.7741
  gbrt: 258.6069
  hist_gbrt: 259.3752
  xgb: 261.3339
  lgbm: 259.8647
  cat: 255.5414
Stacked model OOF RMSE: 254.2667


In [9]:
# 7. Refit Base Models on FULL Training Data

full_pipelines = {}

for name, model in base_models.items():
    print(f"Refitting {name} on full data...")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipe.fit(X, y_log)  # log target
    full_pipelines[name] = pipe

print("All base models refitted on full data.")


Refitting rf on full data...
Refitting gbrt on full data...
Refitting hist_gbrt on full data...
Refitting xgb on full data...
Refitting lgbm on full data...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005321 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3335
[LightGBM] [Info] Number of data points in the train set: 40000, number of used features: 41
[LightGBM] [Info] Start training from score 7.367350
Refitting cat on full data...
All base models refitted on full data.


In [10]:
# 8A. Load Scoring Data & Apply Same Feature Engineering

scoring = pd.read_csv("scoring_data.csv")
X_scoring_base = scoring.drop(columns=["customer_id"])
X_scoring = add_features(X_scoring_base)

print("Scoring data shape (with engineered features):", X_scoring.shape)


Scoring data shape (with engineered features): (10000, 29)


In [11]:
# 8B. Generate Base Model Predictions on Scoring Data
#  (log y → back-transform to original scale)
meta_X_scoring = []

for name in stack_model_names:
    pipe_full = full_pipelines[name]
    scoring_log_pred = pipe_full.predict(X_scoring)
    scoring_pred_raw = np.expm1(scoring_log_pred)
    meta_X_scoring.append(scoring_pred_raw)

meta_X_scoring = np.column_stack(meta_X_scoring)

# Apply stacking meta-learner
scoring_pred_stack = stacker.predict(meta_X_scoring)


In [12]:
# 9. Save Final Kaggle Submission
submission = pd.DataFrame({
    "customer_id": scoring["customer_id"],
    "monthly_spend": scoring_pred_stack
})

submission.to_csv("submission_file_final_optimized_cv_stack.csv", index=False)
print("Saved submission_file_final_optimized_cv_stack.csv")
print("Final OOF stacking RMSE (training data):", round(stack_rmse, 4))


Saved submission_file_final_optimized_cv_stack.csv
Final OOF stacking RMSE (training data): 254.2667


Reflection:
In my analysis, what I did right was progressively strengthening both my feature engineering and my modeling approach. Idid feature engineering to get the variables i needed, for example, total transaction value = number of transactions*average value. I began with simple linear models, but quickly realized they couldn’t capture the complexity in monthly spend. Over time, I identified the strongest numeric predictors, added categorical encoding where appropriate, applied a log transformation to stabilize the target, and introduced cross-validated hyperparameter tuning. I then built a suite of advanced models—Random Forest, Gradient Boosting, XGBoost, and LightGBM—and finally combined them into a stacking ensemble, which produced my best-performing submission.
Where I went wrong was in my early attempts: I used models that were too simple, included weak categorical features that added noise, and created pipelines that became overly complicated or incomplete. Some cells in my workflow were unfinished, and that hurt the consistency between training and scoring.
If I had to do it again, I would establish a clean baseline earlier, simplify preprocessing from the beginning, and introduce model stacking only once the core workflow was stable. I would also explore automated tuning and more targeted feature engineering. Overall, this project showed me how iteration—and learning from failed steps—leads to a much stronger predictive model.

